# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field details with their @id.

record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets in the dataset.\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    print("  Fields:")
    for field in rs['field']:
        print(f"    - {field['@id']} (name: {field.get('name', '')}, type: {field.get('dataType', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into dataframes using their @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if there is data
        dataframes[record_set_id] = pd.DataFrame(records)

# Show available columns for the first record set
if record_set_ids:
    base_record_set_id = record_set_ids[0]
    df = dataframes.get(base_record_set_id, pd.DataFrame())
    print(f"RecordSet @id: {base_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    df.head()
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# -- EDA on the main protein record set --

# You may need to adjust these IDs based on the overview output.
# For this dataset, let's assume one record set stores the main protein data, e.g., '@id': 'cr:ProteinRecordSet'.
# We'll look for a numeric field such as 'cr:MW_kDa' for molecular weight, and group by 'cr:Sample'.

# Find a numeric field to analyze
if dataframes:
    record_set_id = next(iter(dataframes))  # Use the first available dataframe
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    # Try to pick a numeric field by inspecting the columns
    numeric_field = None
    for col in df.columns:
        # Heuristic: pick the first float/int column
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to parse numeric columns from string
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue
    if numeric_field:
        print(f"Using numeric field for analysis: {numeric_field}")

        threshold = df[numeric_field].mean() if df[numeric_field].dtype in ['float64', 'int64'] else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                # Choose a string/categorical column
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframes loaded for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram and boxplot for the numeric field
if dataframes and numeric_field:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")

    plt.tight_layout()
    plt.show()
    # If there is a group_field, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.